# Neural A\* — Parameterised Build

Same flow as `04_fast_track.ipynb`, with the configuration lifted into one cell
so a second or third design can be built without editing anything else. Each run
writes to its own directories, so previous builds are never overwritten.

### The two knobs, and what each controls

| knob | changes | does **not** change |
|---|---|---|
| `CHANNELS` | planning quality, parameter count | — |
| `MAC_BUDGET` | latency and area | planning quality |

Architecture and parallelism are independent. Widening the network improves
`p_exp` and costs area; raising `MAC_BUDGET` improves latency and costs area,
leaving accuracy untouched. Keeping that separation clean is why `REUSE_WEIGHTS`
exists: when only the folding changes, re-training would introduce run-to-run
variance and make it look as though folding affected accuracy. It does not.

### Recommended runs

| tag | CHANNELS | MAC_BUDGET | latency | LUT | purpose |
|---|---|---|---|---|---|
| `""` | (8,16,16) | 34 | 5.58 ms | 45% | already built |
| **`"b"`** | **(8,16,16)** | **54** | **2.79 ms** | **71%** | **2.4x the CPU — run this first** |
| `"c"` | (8,16,32) | 54 | 5.58 ms | 69% | wider network at fixed latency |

Latencies are FINN's single-frame figures, which is the honest metric for a
planner: a robot solves one problem at a time and never fills the pipeline.
16/32/64 is deliberately absent — it reaches only 22.3 ms, over three times
slower than the 6.70 ms software baseline.

## 0. Configuration

In [1]:
# ---- the only cell edited between runs ----------------------------------
TAG          = "b"            # "" was the first build; use a new letter each time
CHANNELS     = (8, 16, 16)    # architecture -> planning quality
MAC_BUDGET   = 54             # parallelism  -> latency and area
REUSE_WEIGHTS = True          # load matching weights if present, rather than retraining
# ---------------------------------------------------------------------------

WEIGHT_BITS, ACT_BITS = 4, 8
CLK_NS = 10.0
PYNQ_Z2 = {"LUT": 53200, "FF": 106400, "BRAM_36K": 140, "DSP": 220}
CPU_MS = 6.70                 # 01_baseline, batch 1, x86

import os, sys, json, glob, time, shutil
import numpy as np
import torch
import torch.nn as nn

WORKDIR = os.getcwd()
os.chdir(WORKDIR)
assert os.path.isdir("neural_astar_core"), f"neural_astar_core not found in {WORKDIR}"
assert os.path.isdir("model"), f"model/ not found in {WORKDIR}"

import brevitas.nn as qnn
from brevitas.quant import Int8ActPerTensorFloat
from brevitas.export import export_qonnx

from neural_astar_core import (sample_problems, load_encoder_from_checkpoint,
                               VanillaAStarPlanner, NeuralAStarPlanner, evaluate)
from neural_astar_core.grid import index_to_one_hot
from neural_astar_core.encoder import encoder_input

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DATA = "mazes_032_moore_c8.npz"
CKPT = "model/mazes_032_moore_c8/lightning_logs/"

ARCH = "_".join(map(str, CHANNELS))
WEIGHTS = f"tiny_encoder_{ARCH}.pt"
ONNX    = os.path.join(WORKDIR, f"tiny_astar{TAG}_w{WEIGHT_BITS}a{ACT_BITS}.onnx")
EST     = os.path.join(WORKDIR, f"tiny_est{TAG}")
EST2    = os.path.join(WORKDIR, f"tiny_est_folded{TAG}")
BUILD   = os.path.join(WORKDIR, f"tiny_output{TAG}")

print(f"tag {TAG!r} | channels {CHANNELS} | budget {MAC_BUDGET} | device {DEVICE}")
print(f"outputs -> tiny_output{TAG}/")

$HOME/miniforge3/envs/fpga_hw2/lib/python3.10/site-packages/brevitas/graph/equalize.py:69: UserWarning: fast_hadamard_transform package not found, using standard pytorch kernels
  warnings.warn("fast_hadamard_transform package not found, using standard pytorch kernels")


tag 'b' | channels (8, 16, 16) | budget 54 | device cuda
outputs -> tiny_outputb/


## 1. Folding plan

In [4]:
def plan_folding(channels, in_dim=2, budget=MAC_BUDGET, pixels=1024):
    """Give parallelism to whichever layer is the current bottleneck.

    Respects all three hardware constraints: SIMD divides the layer's input
    channels, PE divides its output channels, and SIMD >= MW/1024 (an HLS
    MatrixVectorActivation limit that only surfaces at step 11 of the build).
    """
    widths = [in_dim] + list(channels) + [1]
    dims = [(widths[i] * 9, widths[i + 1]) for i in range(len(widths) - 1)]
    ins = [widths[i] for i in range(len(widths) - 1)]
    outs = [widths[i + 1] for i in range(len(widths) - 1)]
    fold = [[1, 1] for _ in dims]

    def cycles(i):
        mw, mh = dims[i]
        return (mw // fold[i][0]) * (mh // fold[i][1]) * pixels

    def used():
        return sum(s * p for s, p in fold)

    while True:
        i = max(range(len(dims)), key=cycles)
        mw, mh = dims[i]
        s, p = fold[i]
        chosen = None
        for ns, np_ in ((s * 2, p), (s, p * 2)):
            ok = (ns <= ins[i] and ins[i] % ns == 0 and mw % ns == 0
                  and np_ <= outs[i] and outs[i] % np_ == 0
                  and ns >= mw / pixels
                  and used() - s * p + ns * np_ <= budget)
            if ok:
                chosen = (ns, np_)
                break
        if chosen is None:
            break
        fold[i] = list(chosen)
    return dims, [tuple(f) for f in fold]


DIMS, FOLD = plan_folding(CHANNELS)
mac_cycle = sum(s * p for s, p in FOLD)
bottleneck = max((mw // s) * (mh // p) * 1024 for (mw, mh), (s, p) in zip(DIMS, FOLD))

# Calibrated on the two designs that reached synthesis: 24,154 LUTs at 27
# MACs/cycle (measured post-synthesis) and 184,839 at 342 (DRC-reported).
est_lut = 10381 + 510 * mac_cycle
# Single-frame latency is ~3.8x the throughput figure, measured on the first build.
LAT_RATIO = 557868 / 147456

print(f"{'conv':6} {'MW':>5} {'MH':>4} {'SIMD':>5} {'PE':>4} {'MAC/cy':>7} {'cycles':>10}")
print("-" * 46)
for i, ((mw, mh), (s, p)) in enumerate(zip(DIMS, FOLD)):
    print(f"conv{i}  {mw:5d} {mh:4d} {s:5d} {p:4d} {s*p:7d} "
          f"{(mw//s)*(mh//p)*1024:10,d}")
print("-" * 46)
print(f"parallelism     : {mac_cycle} MACs/cycle")
print(f"throughput      : {bottleneck*CLK_NS/1e6:.2f} ms/frame")
print(f"single-frame    : {bottleneck*LAT_RATIO*CLK_NS/1e6:.2f} ms   <- the number for a planner")
print(f"estimated LUT   : {est_lut:,.0f} = {100*est_lut/PYNQ_Z2['LUT']:.0f}% of the device")
print(f"vs CPU latency  : {CPU_MS/(bottleneck*LAT_RATIO*CLK_NS/1e6):.2f}x")
if 100*est_lut/PYNQ_Z2['LUT'] > 85:
    print("\nWARNING: projected above 85%. Lower MAC_BUDGET before building.")

conv      MW   MH  SIMD   PE  MAC/cy     cycles
----------------------------------------------
conv0     18    8     2    2       4     36,864
conv1     72   16     8    2      16     73,728
conv2    144   16    16    2      32     73,728
conv3    144    1     2    1       2     73,728
----------------------------------------------
parallelism     : 54 MACs/cycle
throughput      : 0.74 ms/frame
single-frame    : 2.79 ms   <- the number for a planner
estimated LUT   : 37,921 = 71% of the device
vs CPU latency  : 2.40x


## 2. Model

In [5]:
class QuantGuidanceEncoder(nn.Module):
    """Quantized guidance-map encoder at arbitrary width.

    bias=False throughout: every conv is followed by BatchNorm, whose shift
    subsumes any bias, and it leaves FINN one less thing to fold.
    """

    def __init__(self, weight_bits=4, act_bits=8, in_dim=2, channels=CHANNELS):
        super().__init__()
        self.weight_bits, self.act_bits = weight_bits, act_bits
        self.in_dim, self.channels = in_dim, tuple(channels)
        widths = [in_dim] + list(channels) + [1]

        self.input_q = qnn.QuantIdentity(act_quant=Int8ActPerTensorFloat,
                                         return_quant_tensor=True)
        blocks = []
        for i in range(len(widths) - 1):
            blocks.append(qnn.QuantConv2d(
                widths[i], widths[i + 1], 3, 1, 1, bias=False,
                weight_bit_width=weight_bits, return_quant_tensor=False))
            blocks.append(nn.BatchNorm2d(widths[i + 1]))
            if i != len(widths) - 2:
                blocks.append(qnn.QuantReLU(bit_width=act_bits,
                                            return_quant_tensor=True))
        self.model = nn.Sequential(*blocks)
        self.out_q = qnn.QuantIdentity(bit_width=act_bits,
                                       return_quant_tensor=False)

    def forward(self, x):
        return self.out_q(torch.sigmoid(self.model(self.input_q(x))))


float_encoder = load_encoder_from_checkpoint(CKPT, device=DEVICE)
tiny = QuantGuidanceEncoder(WEIGHT_BITS, ACT_BITS, channels=CHANNELS).to(DEVICE)
n_params = sum(p.numel() for p in tiny.parameters())
print(f"float : {sum(p.numel() for p in float_encoder.parameters()):,} params")
print(f"tiny  : {n_params:,} params  {CHANNELS}")

float : 391,395 params
tiny  : 3,831 params  (8, 16, 16)


In [6]:
N_MAPS, STARTS = 800, 4
EPOCHS, BATCH, LR = 120, 64, 3e-3

reused = False
if REUSE_WEIGHTS and os.path.exists(WEIGHTS):
    # Load onto CPU, then move the whole module. Brevitas keeps quantizer scale
    # tensors outside the parts load_state_dict relocates, so loading straight
    # onto the GPU leaves the weight scale on CPU and the first conv fails with
    # "Expected all tensors to be on the same device".
    tiny.load_state_dict(torch.load(WEIGHTS, map_location="cpu", weights_only=True))
    tiny = tiny.to(DEVICE)
    tiny.eval()
    with torch.no_grad():                 # force the quantizers onto DEVICE
        tiny(torch.zeros(1, 2, 32, 32, device=DEVICE))
    reused = True
    print(f"loaded {WEIGHTS} -- skipping training.")
    print("Accuracy is therefore identical to the previous build of this")
    print("architecture, so any change in the results is due to folding alone.")
else:
    def distillation_set(seed=7):
        ps = sample_problems(DATA, "train", N_MAPS * STARTS, seed=seed,
                             starts_per_map=STARTS, replace=True)
        X = torch.cat([encoder_input(p.obstacle_map,
                                     index_to_one_hot(p.start, p.height, p.width),
                                     index_to_one_hot(p.goal, p.height, p.width))
                       for p in ps], 0)
        float_encoder.eval()
        with torch.no_grad():
            Y = torch.cat([float_encoder(X[i:i+256].to(DEVICE)).cpu()
                           for i in range(0, len(X), 256)], 0)
        return X, Y

    t0 = time.time()
    X, Y = distillation_set()
    print(f"{len(X)} instances in {time.time()-t0:.0f}s")

    opt = torch.optim.Adam(tiny.parameters(), lr=LR)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS)
    loss_fn = nn.L1Loss()
    t0 = time.time(); tiny.train()
    for ep in range(EPOCHS):
        perm = torch.randperm(len(X)); total, nb = 0.0, 0
        for i in range(0, len(X), BATCH):
            idx = perm[i:i + BATCH]
            opt.zero_grad()
            loss = loss_fn(tiny(X[idx].to(DEVICE)), Y[idx].to(DEVICE))
            loss.backward(); opt.step()
            total += loss.item(); nb += 1
        sched.step()
        if ep % 20 == 0 or ep == EPOCHS - 1:
            print(f"  epoch {ep+1:4d}/{EPOCHS}  L1 {total/nb:.5f}")
    tiny.eval()
    torch.save(tiny.state_dict(), WEIGHTS)
    print(f"trained in {(time.time()-t0)/60:.1f} min, saved {WEIGHTS}")

$HOME/miniforge3/envs/fpga_hw2/lib/python3.10/site-packages/torch/_tensor.py:1488: UserWarning: Named tensors and all their associated APIs are an experimental feature and subject to change. Please do not use them for anything important until they are released as stable. (Triggered internally at ../c10/core/TensorImpl.h:1928.)
  return super().rename(names)


loaded tiny_encoder_8_16_16.pt -- skipping training.
Accuracy is therefore identical to the previous build of this
architecture, so any change in the results is due to folding alone.


## 3. Planning quality

If weights were reused these numbers reproduce the previous run exactly, which
is the point: it confirms folding did not touch accuracy.

In [7]:
eval_set = sample_problems(DATA, "test", 200, seed=11, starts_per_map=2)
vanilla = VanillaAStarPlanner()

print(f"{'':24} {'p_opt':>7} {'p_exp':>7} {'p_gen':>7}")
rf = evaluate(eval_set, vanilla, NeuralAStarPlanner(float_encoder, device=DEVICE))
print(f"{'float 391k':24} {rf.p_opt:7.3f} {rf.p_exp:7.3f} {rf.p_gen:7.3f}")
rt = evaluate(eval_set, vanilla, NeuralAStarPlanner(tiny, device=DEVICE))
label = f"{ARCH} {n_params//1000}k W{WEIGHT_BITS}A{ACT_BITS}"
print(f"{label:24} {rt.p_opt:7.3f} {rt.p_exp:7.3f} {rt.p_gen:7.3f}")

if rt.p_exp < 0.10:
    print("\nWARNING: p_exp below 0.10 -- the model has barely learned.")
else:
    print(f"\nExpansions cut {rt.p_exp:.0%} vs plain A*.")

json.dump({"tag": TAG, "channels": list(CHANNELS), "budget": MAC_BUDGET,
           "params": n_params, "reused_weights": reused,
           "p_opt": rt.p_opt, "p_exp": rt.p_exp, "p_gen": rt.p_gen,
           "mac_cycle": mac_cycle, "cycles": bottleneck,
           "throughput_ms": bottleneck*CLK_NS/1e6,
           "latency_ms": bottleneck*LAT_RATIO*CLK_NS/1e6,
           "est_lut": est_lut},
          open(f"run{TAG}_summary.json", "w"), indent=1)
print(f"saved run{TAG}_summary.json")

                           p_opt   p_exp   p_gen
float 391k                 0.800   0.441   0.239
8_16_16 3k W4A8            0.750   0.407   0.221

Expansions cut 41% vs plain A*.
saved runb_summary.json


## 4. Export

In [8]:
import onnx
from collections import Counter

tiny.cpu().eval()
export_qonnx(tiny, input_t=torch.zeros(1, 2, 32, 32), export_path=ONNX)

m = onnx.load(ONNX)                 # FINN needs any BipolarQuant scale to be 1.0
bip = {n.input[1] for n in m.graph.node if n.op_type == "BipolarQuant"}
for init in m.graph.initializer:
    if init.name in bip:
        init.raw_data = np.ones(1, dtype=np.float32).tobytes()
        init.float_data[:] = []
onnx.save(m, ONNX)
tiny.to(DEVICE)

print(f"{os.path.basename(ONNX)} ({os.path.getsize(ONNX)//1024} KB)")
print("nodes:", dict(Counter(n.op_type for n in m.graph.node)))

np.savez(f"tiny_test_vectors{TAG}.npz",
         x=(vecs := torch.cat([encoder_input(
              p.obstacle_map,
              index_to_one_hot(p.start, p.height, p.width),
              index_to_one_hot(p.goal, p.height, p.width))
            for p in list(eval_set)[:64]], 0)).numpy(),
         y=tiny(vecs.to(DEVICE)).detach().cpu().numpy())
print(f"saved tiny_test_vectors{TAG}.npz")

tiny_astarb_w4a8.onnx (24 KB)
nodes: {'Quant': 9, 'Conv': 4, 'BatchNormalization': 4, 'Relu': 3, 'Sigmoid': 1}
saved tiny_test_vectorsb.npz


## 5. FINN configuration

In [9]:
import finn.builder.build_dataflow as build
import finn.builder.build_dataflow_config as build_cfg
from qonnx.core.modelwrapper import ModelWrapper
from qonnx.custom_op.registry import getCustomOp
from qonnx.transformation.general import (GiveUniqueNodeNames,
                                          GiveReadableTensorNames)

# Vivado wants libtinfo.so.5; conda ships .so.6. Expose only that symlink rather
# than the whole conda lib/, which would shadow bash's copy and flood the log.
_shim = os.path.expanduser("~/vivado_libs")
os.makedirs(_shim, exist_ok=True)
_t = os.path.join(sys.prefix, "lib", "libtinfo.so.6")
_l = os.path.join(_shim, "libtinfo.so.5")
if os.path.exists(_t) and not os.path.exists(_l):
    os.symlink(_t, _l)
os.environ["LD_LIBRARY_PATH"] = _shim + ":" + os.environ.get("LD_LIBRARY_PATH", "")
os.environ["FINN_ROOT"] = os.path.expanduser("~/finn")


def step_name_nodes(model, cfg):
    # Brevitas exports unnamed nodes and step_specialize_layers creates more.
    # Unnamed layers collapse the estimate reports to one blank key and make the
    # folding config unaddressable.
    model = model.transform(GiveUniqueNodeNames())
    return model.transform(GiveReadableTensorNames())


def _with_naming(steps):
    steps = list(steps)
    steps.insert(steps.index("step_specialize_layers") + 1, step_name_nodes)
    return steps


def _estimate_steps():
    s = getattr(build_cfg, "estimate_only_dataflow_steps", None)
    if s:
        return list(s)
    full = build_cfg.default_build_dataflow_steps
    return list(full[:full.index("step_generate_estimate_reports") + 1])


ESTIMATE_STEPS = _with_naming(_estimate_steps())
FULL_STEPS = _with_naming(build_cfg.default_build_dataflow_steps)

SPECIALIZE = os.path.join(WORKDIR, f"specialize{TAG}.json")
json.dump({"Defaults": {"preferred_impl_style": ["rtl", "all"]}},
          open(SPECIALIZE, "w"), indent=2)


def make_cfg(output_dir, build_dir, folding=None, bitfile=False):
    os.environ["FINN_BUILD_DIR"] = build_dir
    os.makedirs(build_dir, exist_ok=True)
    outs = [build_cfg.DataflowOutputType.ESTIMATE_REPORTS]
    if bitfile:
        outs += [build_cfg.DataflowOutputType.BITFILE,
                 build_cfg.DataflowOutputType.PYNQ_DRIVER,
                 build_cfg.DataflowOutputType.DEPLOYMENT_PACKAGE]
    return build_cfg.DataflowBuildConfig(
        output_dir=output_dir, board="Pynq-Z2",
        shell_flow_type=build_cfg.ShellFlowType.VIVADO_ZYNQ,
        synth_clk_period_ns=CLK_NS, generate_outputs=outs,
        folding_config_file=folding,
        specialize_layers_config_file=SPECIALIZE,
        auto_fifo_depths=False, verify_save_rtlsim_waveforms=False,
        steps=FULL_STEPS if bitfile else ESTIMATE_STEPS)


def fresh(*dirs):
    for d in dirs:
        if os.path.exists(d):
            shutil.rmtree(d, ignore_errors=True)


def load_report(output_dir, name):
    p = os.path.join(output_dir, "report", name)
    return json.load(open(p)) if os.path.exists(p) else None


print(f"estimate flow {len(ESTIMATE_STEPS)} steps | full flow {len(FULL_STEPS)} steps")

estimate flow 11 steps | full flow 20 steps


## 6. Estimate build, then folding

In [18]:
fresh(EST, EST + "_tmp")
t0 = time.time()
build.build_dataflow_cfg(ONNX, make_cfg(EST, EST + "_tmp"))
print(f"\nestimate build in {time.time()-t0:.0f}s")

Building dataflow accelerator from $HOME/project/tiny_astarb_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/tiny_estb_tmp
Final outputs will be generated in $HOME/project/tiny_estb
Build log is at $HOME/project/tiny_estb/build_dataflow.log
Running step: step_qonnx_to_finn [1/11]
Running step: step_tidy_up [2/11]
Running step: step_streamline [3/11]
Running step: step_convert_to_hw [4/11]
Running step: step_create_dataflow_partition [5/11]
Running step: step_specialize_layers [6/11]
Running step: step_name_nodes [7/11]
Running step: step_target_fps_parallelization [8/11]
Running step: step_apply_folding_config [9/11]
Running step: step_minimize_bit_width [10/11]
Running step: step_generate_estimate_reports [11/11]
Completed successfully

estimate build in 2s


In [19]:
def discover_layers(output_dir):
    # FINN writes intermediate_models/<step>.onnx AFTER each step, so the
    # specialize snapshot predates step_name_nodes and its nodes are unnamed.
    # Scan newest-first for one that has names; otherwise derive them the way
    # GiveUniqueNodeNames does -- {op_type}_{per-type counter}, graph order.
    hits = glob.glob(os.path.join(output_dir, "intermediate_models", "*.onnx"))
    if not hits:
        raise RuntimeError("no intermediate models -- estimate build failed?")
    hits.sort(key=os.path.getmtime)

    def mvaus(path):
        model = ModelWrapper(path)
        out = []
        for n in model.graph.node:
            if "MVAU" not in n.op_type and "VVAU" not in n.op_type:
                continue
            try:
                inst = getCustomOp(n)
                out.append((n.name, n.op_type,
                            inst.get_nodeattr("MW"), inst.get_nodeattr("MH")))
            except Exception:
                out.append((n.name, n.op_type, None, None))
        return out

    for path in reversed(hits):
        got = mvaus(path)
        if got and all(n for n, _, _, _ in got):
            return os.path.basename(path), got
    for path in reversed(hits):
        got = mvaus(path)
        if got:
            ctr, named = {}, []
            for _, op, mw, mh in got:
                i = ctr.get(op, 0); ctr[op] = i + 1
                named.append((f"{op}_{i}", op, mw, mh))
            return os.path.basename(path) + " (derived)", named
    raise RuntimeError("no MVAU nodes found")


def write_folding(output_dir, path):
    src, layers = discover_layers(output_dir)
    print(f"layers from {src}\n")
    by_dim = {d: f for d, f in zip(DIMS, FOLD)}
    cfg = {"Defaults": {}}
    print(f"{'node':<22} {'op_type':<12} {'MW':>5} {'MH':>4} {'SIMD':>5} {'PE':>4}")
    print("-" * 58)
    for i, (name, op, mw, mh) in enumerate(layers):
        f = by_dim.get((mw, mh)) or (FOLD[i] if i < len(FOLD) else None)
        if f is None:
            continue
        cfg[name] = {"SIMD": f[0], "PE": f[1]}
        print(f"{name:<22} {op:<12} {str(mw):>5} {str(mh):>4} {f[0]:5d} {f[1]:4d}")
    missing = [n for n, _, _, _ in layers if n not in cfg]
    if missing:
        raise RuntimeError(f"no folding for {missing} -- would run at SIMD=1 "
                           "and fail HLS codegen")
    json.dump(cfg, open(path, "w"), indent=2)
    print(f"\nwrote {os.path.basename(path)} ({len(cfg)-1} layers)")
    return cfg


FOLDING = os.path.join(WORKDIR, f"folding{TAG}.json")
write_folding(EST, FOLDING)

layers from step_generate_estimate_reports.onnx

node                   op_type         MW   MH  SIMD   PE
----------------------------------------------------------
MVAU_hls_0             MVAU_hls        18    8     2    2
MVAU_hls_1             MVAU_hls        72   16     8    2
MVAU_hls_2             MVAU_hls       144   16    16    2
MVAU_rtl_0             MVAU_rtl       144    1     2    1

wrote foldingb.json (4 layers)


{'Defaults': {},
 'MVAU_hls_0': {'SIMD': 2, 'PE': 2},
 'MVAU_hls_1': {'SIMD': 8, 'PE': 2},
 'MVAU_hls_2': {'SIMD': 16, 'PE': 2},
 'MVAU_rtl_0': {'SIMD': 2, 'PE': 1}}

In [20]:
fresh(EST2, EST2 + "_tmp")
t0 = time.time()
build.build_dataflow_cfg(ONNX, make_cfg(EST2, EST2 + "_tmp", folding=FOLDING))
print(f"done in {time.time()-t0:.0f}s\n")

cyc = load_report(EST2, "estimate_layer_cycles.json")
res = load_report(EST2, "estimate_layer_resources.json")

if cyc:
    worst = max(cyc.values())
    for k, v in sorted(cyc.items(), key=server kv: -kv[1])[:8]:
        print(f"  {k:<34} {v:10,d}")
    print(f"\nthroughput   : {worst*CLK_NS/1e6:.2f} ms")
    print(f"single-frame : {worst*LAT_RATIO*CLK_NS/1e6:.2f} ms")
    print(f"vs CPU       : {CPU_MS/(worst*LAT_RATIO*CLK_NS/1e6):.2f}x")

if res:
    t = res.get("total", {})
    lut = t.get("LUT", 0)
    print(f"\nFINN estimate: {lut:,.0f} LUT")
    print(f"Calibrated   : {est_lut:,.0f} LUT = {100*est_lut/PYNQ_Z2['LUT']:.0f}%")
    print("\nFINN's own estimate ran 5.4x low on the first build; trust the")
    print("calibrated figure, which is anchored on two measured designs.")
    if 100*est_lut/PYNQ_Z2["LUT"] > 85:
        print("\nOVER 85% -- lower MAC_BUDGET and re-run from section 0.")
    else:
        print("\nProceed to the full build.")

Building dataflow accelerator from $HOME/project/tiny_astarb_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/tiny_est_foldedb_tmp
Final outputs will be generated in $HOME/project/tiny_est_foldedb
Build log is at $HOME/project/tiny_est_foldedb/build_dataflow.log
Running step: step_qonnx_to_finn [1/11]
Running step: step_tidy_up [2/11]
Running step: step_streamline [3/11]
Running step: step_convert_to_hw [4/11]
Running step: step_create_dataflow_partition [5/11]
Running step: step_specialize_layers [6/11]
Running step: step_name_nodes [7/11]
Running step: step_target_fps_parallelization [8/11]
Running step: step_apply_folding_config [9/11]
Running step: step_minimize_bit_width [10/11]
Running step: step_generate_estimate_reports [11/11]
Completed successfully
done in 2s

  MVAU_hls_1                             73,728
  MVAU_hls_2                             73,728
  MVAU_rtl_0                             73,728
  MVAU_hls_0                             36,864
  Convolut

## 7. Full build

30-60 minutes. **Use a long allocation** — an interactive `srun` that expires
takes the build with it, and tmux does not help because Slurm kills the whole
job step:

```bash
srun --gres=gpu:1 --time=12:00:00 --mem=32G --cpus-per-task=2 --pty bash
```

Progress does not always reach the notebook; watch the filesystem instead:

```bash
watch -n 60 'ls -lt tiny_output*/intermediate_models/ | head -3'
```

In [21]:
fresh(BUILD, BUILD + "_tmp")
print(f"full build started {time.strftime('%H:%M:%S')} -- 30-60 min\n")
t0 = time.time()
build.build_dataflow_cfg(ONNX, make_cfg(BUILD, BUILD + "_tmp",
                                        folding=FOLDING, bitfile=True))
print(f"\nBUILD COMPLETE in {(time.time()-t0)/60:.0f} min")

full build started 18:00:44 -- 30-60 min

Building dataflow accelerator from $HOME/project/tiny_astarb_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/tiny_outputb_tmp
Final outputs will be generated in $HOME/project/tiny_outputb
Build log is at $HOME/project/tiny_outputb/build_dataflow.log
Running step: step_qonnx_to_finn [1/20]
Running step: step_tidy_up [2/20]
Running step: step_streamline [3/20]
Running step: step_convert_to_hw [4/20]
Running step: step_create_dataflow_partition [5/20]
Running step: step_specialize_layers [6/20]
Running step: step_name_nodes [7/20]
Running step: step_target_fps_parallelization [8/20]
Running step: step_apply_folding_config [9/20]
Running step: step_minimize_bit_width [10/20]
Running step: step_generate_estimate_reports [11/20]
Running step: step_hw_codegen [12/20]
Running step: step_hw_ipgen [13/20]
Running step: step_set_fifo_depths [14/20]
Running step: step_create_stitched_ip [15/20]
Running step: step_measure_rtlsim_performanc

In [22]:
top = (load_report(BUILD, "post_synth_resources.json") or {}).get("(top)", {})
summary = json.load(open(f"run{TAG}_summary.json"))

if top:
    print(f"{'Resource':<12} {'Used':>9} {'Available':>11} {'Util':>8}")
    print("-" * 44)
    for key, avail in PYNQ_Z2.items():
        used = top.get(key, top.get(key.replace("_36K", ""), 0))
        print(f"{key:<12} {used:9.0f} {avail:11d} {100*used/avail:7.1f}%")
    lut = top.get("LUT", 0)
    print(f"\npredicted {est_lut:,.0f} LUT, measured {lut:,.0f} "
          f"({lut/est_lut:.2f}x)")
    summary["measured_lut"] = lut
    summary["measured_lut_pct"] = 100 * lut / PYNQ_Z2["LUT"]

perf = load_report(BUILD, "estimate_network_performance.json")
if perf:
    print(f"\nthroughput   : {1000/perf['estimated_throughput_fps']:.2f} ms/frame")
    print(f"single-frame : {perf['estimated_latency_ns']/1e6:.2f} ms")
    print(f"vs CPU       : {CPU_MS/(perf['estimated_latency_ns']/1e6):.2f}x")
    summary["finn_latency_ms"] = perf["estimated_latency_ns"] / 1e6
    summary["finn_throughput_ms"] = 1000 / perf["estimated_throughput_fps"]

rpt = os.path.join(BUILD, "report", "post_route_timing.rpt")
if os.path.exists(rpt):
    for line in open(rpt):
        if "All user specified timing constraints are met" in line:
            print("\ntiming: MET"); summary["timing_met"] = True; break
    else:
        print("\ntiming: CHECK post_route_timing.rpt"); summary["timing_met"] = False

json.dump(summary, open(f"run{TAG}_summary.json", "w"), indent=1)

deploy = os.path.join(BUILD, "deploy")
if os.path.isdir(deploy):
    print(f"\nDEPLOYMENT PACKAGE: {deploy}")
    print("\nCopy to the board:")
    print(f"  scp -r {os.path.basename(BUILD)}/deploy xilinx@<board-ip>:~/astar{TAG}")
    print(f"  scp tiny_test_vectors{TAG}.npz mazes_032_moore_c8.npz "
          f"xilinx@<board-ip>:~/astar{TAG}/")
    print(f"  scp -r neural_astar_core 05_board.ipynb xilinx@<board-ip>:~/astar{TAG}/")

Resource          Used   Available     Util
--------------------------------------------
LUT              33820       53200    63.6%
FF               31058      106400    29.2%
BRAM_36K             5         140     3.6%
DSP                 60         220    27.3%

predicted 37,921 LUT, measured 33,820 (0.89x)

throughput   : 0.74 ms/frame
single-frame : 3.00 ms
vs CPU       : 2.23x

timing: MET

DEPLOYMENT PACKAGE: $HOME/project/tiny_outputb/deploy

Copy to the board:
  scp -r tiny_outputb/deploy xilinx@<board-ip>:~/astarb
  scp tiny_test_vectorsb.npz mazes_032_moore_c8.npz xilinx@<board-ip>:~/astarb/
  scp -r neural_astar_core 05_board.ipynb xilinx@<board-ip>:~/astarb/


## 8. Comparing runs

Run this after any build to see all completed configurations side by side.

In [29]:
rows = []
for f in sorted(glob.glob("run*_summary.json")):
    s = json.load(open(f))
    rows.append(s)

if rows:
    print(f"{'tag':>4} {'channels':>10} {'budget':>7} {'p_exp':>7} {'p_gen':>7} "
          f"{'lat ms':>7} {'vs CPU':>7} {'LUT%':>7}")
    print("-" * 64)
    for s in rows:
        lut = s.get("measured_lut_pct", 100*s["est_lut"]/PYNQ_Z2["LUT"])
        star = "" if "measured_lut_pct" in s else " (est)"
        lat = s.get("finn_latency_ms", s["latency_ms"])
        print(f"{s['tag'] or '-':>4} {'/'.join(map(str,s['channels'])):>10} "
              f"{s['budget']:7d} {s['p_exp']:7.3f} {s['p_gen']:7.3f} "
              f"{lat:7.2f} {CPU_MS/lat:6.2f}x {lut:6.0f}%{star}")
    print("\nArchitecture sets p_exp and p_gen; MAC_BUDGET sets latency and area.")
    print("They are independent, which is why reusing weights matters when only")
    print("the folding changes.")

 tag   channels  budget   p_exp   p_gen  lat ms  vs CPU    LUT%
----------------------------------------------------------------
   b    8/16/16      54   0.407   0.221    3.00   2.23x     64%
   d    8/16/16      54   0.412   0.228    2.79   2.40x     66%

Architecture sets p_exp and p_gen; MAC_BUDGET sets latency and area.
They are independent, which is why reusing weights matters when only
the folding changes.


In [26]:

# ---------------------------------------------------------------- CELL D1 ---
# Train the no-sigmoid variant, evaluate it, export. ~5 minutes on GPU.

TAG_D = "d"
ONNX_D  = os.path.join(WORKDIR, f"tiny_astar{TAG_D}_w{WEIGHT_BITS}a{ACT_BITS}.onnx")
EST_D   = os.path.join(WORKDIR, f"tiny_est{TAG_D}")
EST2_D  = os.path.join(WORKDIR, f"tiny_est_folded{TAG_D}")
BUILD_D = os.path.join(WORKDIR, f"tiny_output{TAG_D}")


class QuantEncoderNoSigmoid(QuantGuidanceEncoder):
    """Same network, sigmoid moved to the host.

    forward() returns the pre-sigmoid value. Apply torch.sigmoid() to whatever
    the accelerator returns to recover the guidance map.
    """

    def forward(self, x):
        return self.out_q(self.model(self.input_q(x)))


class HostSigmoid(nn.Module):
    """Wrapper so the A* metrics stay comparable with the other builds."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return torch.sigmoid(self.m(x))


tiny_d = QuantEncoderNoSigmoid(WEIGHT_BITS, ACT_BITS, channels=CHANNELS).to(DEVICE)
print(f"{sum(p.numel() for p in tiny_d.parameters()):,} params, {CHANNELS}, no sigmoid")

# Targets are now pre-sigmoid, so take them from the float encoder's body.
ps = sample_problems(DATA, "train", 3200, seed=7, starts_per_map=4, replace=True)
Xd = torch.cat([encoder_input(p.obstacle_map,
                              index_to_one_hot(p.start, p.height, p.width),
                              index_to_one_hot(p.goal, p.height, p.width))
                for p in ps], 0)
float_encoder.eval()
with torch.no_grad():
    Yd = torch.cat([float_encoder.model(Xd[i:i+256].to(DEVICE)).cpu()
                    for i in range(0, len(Xd), 256)], 0)
print(f"{len(Xd)} instances | target range [{Yd.min():.2f}, {Yd.max():.2f}] (pre-sigmoid)")

opt = torch.optim.Adam(tiny_d.parameters(), lr=3e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=120)
loss_fn = nn.L1Loss()
t0 = time.time(); tiny_d.train()
for ep in range(120):
    perm = torch.randperm(len(Xd)); total, nb = 0.0, 0
    for i in range(0, len(Xd), 64):
        idx = perm[i:i + 64]
        opt.zero_grad()
        loss = loss_fn(tiny_d(Xd[idx].to(DEVICE)), Yd[idx].to(DEVICE))
        loss.backward(); opt.step()
        total += loss.item(); nb += 1
    sched.step()
    if ep % 20 == 0 or ep == 119:
        print(f"  epoch {ep+1:4d}/120  L1 {total/nb:.5f}")
tiny_d.eval()
torch.save(tiny_d.state_dict(), f"tiny_encoder_{ARCH}_nosigmoid.pt")
print(f"trained in {(time.time()-t0)/60:.1f} min")

rd = evaluate(eval_set, vanilla, NeuralAStarPlanner(HostSigmoid(tiny_d), device=DEVICE))
print(f"\n{'':24} {'p_opt':>7} {'p_exp':>7} {'p_gen':>7}")
print(f"{'build B (sigmoid on HW)':24} {0.750:7.3f} {0.407:7.3f} {0.221:7.3f}")
print(f"{'build D (sigmoid on host)':24} {rd.p_opt:7.3f} {rd.p_exp:7.3f} {rd.p_gen:7.3f}")
print("\nThese should be close. A large gap means the pre-sigmoid targets are")
print("harder to fit than the squashed ones -- raise epochs if so.")

# --- export ---
tiny_d.cpu().eval()
export_qonnx(tiny_d, input_t=torch.zeros(1, 2, 32, 32), export_path=ONNX_D)
m = onnx.load(ONNX_D)
bip = {n.input[1] for n in m.graph.node if n.op_type == "BipolarQuant"}
for init in m.graph.initializer:
    if init.name in bip:
        init.raw_data = np.ones(1, dtype=np.float32).tobytes()
        init.float_data[:] = []
onnx.save(m, ONNX_D)
tiny_d.to(DEVICE)

from collections import Counter
ops = Counter(n.op_type for n in m.graph.node)
print(f"\n{os.path.basename(ONNX_D)}  nodes: {dict(ops)}")
print("Sigmoid in graph:", ops.get("Sigmoid", 0), "(want 0)")

vecs = torch.cat([encoder_input(p.obstacle_map,
                                index_to_one_hot(p.start, p.height, p.width),
                                index_to_one_hot(p.goal, p.height, p.width))
                  for p in list(eval_set)[:64]], 0)
np.savez(f"tiny_test_vectors{TAG_D}.npz",
         x=vecs.numpy(),
         y=torch.sigmoid(tiny_d(vecs.to(DEVICE))).detach().cpu().numpy())
print(f"saved tiny_test_vectors{TAG_D}.npz (y is post-sigmoid, as before)")

3,831 params, (8, 16, 16), no sigmoid
3200 instances | target range [-9.13, 9.67] (pre-sigmoid)
  epoch    1/120  L1 0.89261
  epoch   21/120  L1 0.49852
  epoch   41/120  L1 0.46924
  epoch   61/120  L1 0.46347
  epoch   81/120  L1 0.45004
  epoch  101/120  L1 0.44177
  epoch  120/120  L1 0.42717
trained in 2.4 min

                           p_opt   p_exp   p_gen
build B (sigmoid on HW)    0.750   0.407   0.221
build D (sigmoid on host)   0.655   0.412   0.228

These should be close. A large gap means the pre-sigmoid targets are
harder to fit than the squashed ones -- raise epochs if so.

tiny_astard_w4a8.onnx  nodes: {'Quant': 9, 'Conv': 4, 'BatchNormalization': 4, 'Relu': 3}
Sigmoid in graph: 0 (want 0)
saved tiny_test_vectorsd.npz (y is post-sigmoid, as before)


In [27]:

# ---------------------------------------------------------------- CELL D2 ---
# Estimate build and folding. ~2 minutes. Check odt before going further.

fresh(EST_D, EST_D + "_tmp")
t0 = time.time()
build.build_dataflow_cfg(ONNX_D, make_cfg(EST_D, EST_D + "_tmp"))
print(f"estimate build in {time.time()-t0:.0f}s\n")

FOLDING_D = os.path.join(WORKDIR, f"folding{TAG_D}.json")
write_folding(EST_D, FOLDING_D)

fresh(EST2_D, EST2_D + "_tmp")
build.build_dataflow_cfg(ONNX_D, make_cfg(EST2_D, EST2_D + "_tmp", folding=FOLDING_D))

cyc = load_report(EST2_D, "estimate_layer_cycles.json")
if cyc:
    worst = max(cyc.values())
    print(f"\nthroughput   : {worst*CLK_NS/1e6:.2f} ms")
    print(f"single-frame : {worst*(557868/147456)*CLK_NS/1e6:.2f} ms")

# The payoff is visible here: the deployment driver reports the output type.
drv = glob.glob(os.path.join(EST2_D, "**", "driver.py"), recursive=True)
if drv:
    txt = open(drv[0]).read()
    for key in ("odt", "oshape_normal"):
        for line in txt.splitlines():
            if line.strip().startswith(key):
                print(f"  {line.strip()}")
print("\nWant odt to be UINT8/INT8 rather than INT24. If it still says INT24,")
print("FINN did not fold the output quantiser and this rebuild will not help.")



Building dataflow accelerator from $HOME/project/tiny_astard_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/tiny_estd_tmp
Final outputs will be generated in $HOME/project/tiny_estd
Build log is at $HOME/project/tiny_estd/build_dataflow.log
Running step: step_qonnx_to_finn [1/11]
Running step: step_tidy_up [2/11]
Running step: step_streamline [3/11]
Running step: step_convert_to_hw [4/11]
Running step: step_create_dataflow_partition [5/11]
Running step: step_specialize_layers [6/11]
Running step: step_name_nodes [7/11]
Running step: step_target_fps_parallelization [8/11]
Running step: step_apply_folding_config [9/11]
Running step: step_minimize_bit_width [10/11]
Running step: step_generate_estimate_reports [11/11]
Completed successfully
estimate build in 2s

layers from step_generate_estimate_reports.onnx

node                   op_type         MW   MH  SIMD   PE
----------------------------------------------------------
MVAU_hls_0             MVAU_hls        18    8 

In [28]:

# ---------------------------------------------------------------- CELL D3 ---
# Full build. 30-60 min. Run only if D2 showed an 8-bit output type.

fresh(BUILD_D, BUILD_D + "_tmp")
print(f"full build started {time.strftime('%H:%M:%S')}\n")
t0 = time.time()
build.build_dataflow_cfg(ONNX_D, make_cfg(BUILD_D, BUILD_D + "_tmp",
                                          folding=FOLDING_D, bitfile=True))
print(f"\nBUILD COMPLETE in {(time.time()-t0)/60:.0f} min")

top = (load_report(BUILD_D, "post_synth_resources.json") or {}).get("(top)", {})
if top:
    print(f"\n{'Resource':<12} {'Used':>9} {'Available':>11} {'Util':>8}")
    print("-" * 44)
    for key, avail in PYNQ_Z2.items():
        used = top.get(key, top.get(key.replace("_36K", ""), 0))
        print(f"{key:<12} {used:9.0f} {avail:11d} {100*used/avail:7.1f}%")
    print(f"\nbuild B measured 33,820 LUT (63.6%) for comparison")

perf = load_report(BUILD_D, "estimate_network_performance.json")
if perf:
    print(f"\nthroughput   : {1000/perf['estimated_throughput_fps']:.2f} ms/frame")
    print(f"single-frame : {perf['estimated_latency_ns']/1e6:.2f} ms")
    print(f"vs CPU       : {CPU_MS/(perf['estimated_latency_ns']/1e6):.2f}x")

json.dump({"tag": TAG_D, "channels": list(CHANNELS), "budget": MAC_BUDGET,
           "params": sum(p.numel() for p in tiny_d.parameters()),
           "no_sigmoid": True, "reused_weights": False,
           "p_opt": rd.p_opt, "p_exp": rd.p_exp, "p_gen": rd.p_gen,
           "mac_cycle": mac_cycle, "cycles": bottleneck,
           "throughput_ms": bottleneck*CLK_NS/1e6,
           "latency_ms": bottleneck*(557868/147456)*CLK_NS/1e6,
           "est_lut": est_lut,
           "measured_lut": top.get("LUT", 0),
           "measured_lut_pct": 100*top.get("LUT", 0)/PYNQ_Z2["LUT"]},
          open(f"run{TAG_D}_summary.json", "w"), indent=1)

deploy = os.path.join(BUILD_D, "deploy")
if os.path.isdir(deploy):
    print(f"\nDEPLOYMENT PACKAGE: {deploy}")
    print("\nOn the board, section 3's calibration should now find a much")
    print("smaller scale factor, and section 4's latency should drop by two")
    print("orders of magnitude. Copy with:")
    print(f"  scp -r tiny_output{TAG_D}/deploy xilinx@<board-ip>:~/astar{TAG_D}")
    print(f"  scp tiny_test_vectors{TAG_D}.npz mazes_032_moore_c8.npz "
          f"xilinx@<board-ip>:~/astar{TAG_D}/")
    print(f"  scp -r neural_astar_core 05_board.ipynb xilinx@<board-ip>:~/astar{TAG_D}/")


full build started 21:56:39

Building dataflow accelerator from $HOME/project/tiny_astard_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/tiny_outputd_tmp
Final outputs will be generated in $HOME/project/tiny_outputd
Build log is at $HOME/project/tiny_outputd/build_dataflow.log
Running step: step_qonnx_to_finn [1/20]
Running step: step_tidy_up [2/20]
Running step: step_streamline [3/20]
Running step: step_convert_to_hw [4/20]
Running step: step_create_dataflow_partition [5/20]
Running step: step_specialize_layers [6/20]
Running step: step_name_nodes [7/20]
Running step: step_target_fps_parallelization [8/20]
Running step: step_apply_folding_config [9/20]
Running step: step_minimize_bit_width [10/20]
Running step: step_generate_estimate_reports [11/20]
Running step: step_hw_codegen [12/20]
Running step: step_hw_ipgen [13/20]
Running step: step_set_fifo_depths [14/20]
Running step: step_create_stitched_ip [15/20]
Running step: step_measure_rtlsim_performance [16/20]
Run

In [ ]:
NEW STEPS


In [11]:
class QuantEncoderNoSigmoid(QuantGuidanceEncoder):
    """Same network, sigmoid moved to the host.

    forward() returns the pre-sigmoid value. Apply torch.sigmoid() to whatever
    the accelerator returns to recover the guidance map.
    """

    def forward(self, x):
        return self.out_q(self.model(self.input_q(x)))


class HostSigmoid(nn.Module):
    """Wrapper so the A* metrics stay comparable with the other builds."""

    def __init__(self, m):
        super().__init__()
        self.m = m

    def forward(self, x):
        return torch.sigmoid(self.m(x))


print("defined:", QuantEncoderNoSigmoid.__name__, "and", HostSigmoid.__name__)

defined: QuantEncoderNoSigmoid and HostSigmoid


In [12]:

# ---------------------------------------------------------------- CELL E1 ---
# Retrain with clamped targets and evaluate. ~3 minutes. Stop if p_opt
# does not improve -- there is no point building a worse version of D.

TAG_E = "e"
CLAMP = 6.0
ONNX_E  = os.path.join(WORKDIR, f"tiny_astar{TAG_E}_w{WEIGHT_BITS}a{ACT_BITS}.onnx")
EST_E   = os.path.join(WORKDIR, f"tiny_est{TAG_E}")
EST2_E  = os.path.join(WORKDIR, f"tiny_est_folded{TAG_E}")
BUILD_E = os.path.join(WORKDIR, f"tiny_output{TAG_E}")

tiny_e = QuantEncoderNoSigmoid(WEIGHT_BITS, ACT_BITS, channels=CHANNELS).to(DEVICE)

ps = sample_problems(DATA, "train", 3200, seed=7, starts_per_map=4, replace=True)
Xe = torch.cat([encoder_input(p.obstacle_map,
                              index_to_one_hot(p.start, p.height, p.width),
                              index_to_one_hot(p.goal, p.height, p.width))
                for p in ps], 0)
float_encoder.eval()
with torch.no_grad():
    Ye = torch.cat([float_encoder.model(Xe[i:i+256].to(DEVICE)).cpu()
                    for i in range(0, len(Xe), 256)], 0)

print(f"target range before clamp : [{Ye.min():.2f}, {Ye.max():.2f}]")
Ye = torch.clamp(Ye, -CLAMP, CLAMP)          # <-- the only change from variant D
print(f"target range after  clamp : [{Ye.min():.2f}, {Ye.max():.2f}]")
print(f"sigmoid({CLAMP}) = {1/(1+np.exp(-CLAMP)):.4f}, so the clipped tails are")
print("saturated and carry no information the search can use.\n")

opt = torch.optim.Adam(tiny_e.parameters(), lr=3e-3)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=120)
loss_fn = nn.L1Loss()
t0 = time.time(); tiny_e.train()
for ep in range(120):
    perm = torch.randperm(len(Xe)); total, nb = 0.0, 0
    for i in range(0, len(Xe), 64):
        idx = perm[i:i + 64]
        opt.zero_grad()
        loss = loss_fn(tiny_e(Xe[idx].to(DEVICE)), Ye[idx].to(DEVICE))
        loss.backward(); opt.step()
        total += loss.item(); nb += 1
    sched.step()
    if ep % 20 == 0 or ep == 119:
        print(f"  epoch {ep+1:4d}/120  L1 {total/nb:.5f}")
final_l1 = total / nb
tiny_e.eval()
torch.save(tiny_e.state_dict(), f"tiny_encoder_{ARCH}_nosigmoid_clamped.pt")
print(f"trained in {(time.time()-t0)/60:.1f} min")

re_ = evaluate(eval_set, vanilla, NeuralAStarPlanner(HostSigmoid(tiny_e), device=DEVICE))
print(f"\n{'':30} {'L1':>8} {'p_opt':>7} {'p_exp':>7} {'p_gen':>7}")
print(f"{'B  sigmoid on hardware':30} {0.009:8.3f} {0.750:7.3f} {0.407:7.3f} {0.221:7.3f}")
print(f"{'D  host sigmoid, unclamped':30} {0.427:8.3f} {0.655:7.3f} {0.412:7.3f} {0.228:7.3f}")
print(f"{'E  host sigmoid, clamped':30} {final_l1:8.3f} {re_.p_opt:7.3f} "
      f"{re_.p_exp:7.3f} {re_.p_gen:7.3f}")

if re_.p_opt > 0.72:
    print("\nRecovered. Build E gives D's interface with B's optimality --")
    print("continue to E2.")
elif re_.p_opt > 0.655:
    print("\nPartial recovery. Worth building if time allows, but the")
    print("trade-off described in the report still stands.")
else:
    print("\nNo recovery. Do not build: keep D and leave the report's account")
    print("of the trade-off as written. Try CLAMP = 4.0 first instead.")


target range before clamp : [-9.13, 9.67]
target range after  clamp : [-6.00, 6.00]
sigmoid(6.0) = 0.9975, so the clipped tails are
saturated and carry no information the search can use.

  epoch    1/120  L1 0.91066
  epoch   21/120  L1 0.49835
  epoch   41/120  L1 0.46431
  epoch   61/120  L1 0.45233
  epoch   81/120  L1 0.44366
  epoch  101/120  L1 0.43808
  epoch  120/120  L1 0.42436
trained in 2.4 min

                                     L1   p_opt   p_exp   p_gen
B  sigmoid on hardware            0.009   0.750   0.407   0.221
D  host sigmoid, unclamped        0.427   0.655   0.412   0.228
E  host sigmoid, clamped          0.424   0.680   0.335   0.188

Partial recovery. Worth building if time allows, but the
trade-off described in the report still stands.


In [14]:
def discover_layers(output_dir):
    # FINN writes intermediate_models/<step>.onnx AFTER each step, so the
    # specialize snapshot predates step_name_nodes and its nodes are unnamed.
    # Scan newest-first for one that has names; otherwise derive them the way
    # GiveUniqueNodeNames does -- {op_type}_{per-type counter}, graph order.
    hits = glob.glob(os.path.join(output_dir, "intermediate_models", "*.onnx"))
    if not hits:
        raise RuntimeError("no intermediate models -- estimate build failed?")
    hits.sort(key=os.path.getmtime)

    def mvaus(path):
        model = ModelWrapper(path)
        out = []
        for n in model.graph.node:
            if "MVAU" not in n.op_type and "VVAU" not in n.op_type:
                continue
            try:
                inst = getCustomOp(n)
                out.append((n.name, n.op_type,
                            inst.get_nodeattr("MW"), inst.get_nodeattr("MH")))
            except Exception:
                out.append((n.name, n.op_type, None, None))
        return out

    for path in reversed(hits):
        got = mvaus(path)
        if got and all(n for n, _, _, _ in got):
            return os.path.basename(path), got
    for path in reversed(hits):
        got = mvaus(path)
        if got:
            ctr, named = {}, []
            for _, op, mw, mh in got:
                i = ctr.get(op, 0); ctr[op] = i + 1
                named.append((f"{op}_{i}", op, mw, mh))
            return os.path.basename(path) + " (derived)", named
    raise RuntimeError("no MVAU nodes found")


def write_folding(output_dir, path):
    src, layers = discover_layers(output_dir)
    print(f"layers from {src}\n")
    by_dim = {d: f for d, f in zip(DIMS, FOLD)}
    cfg = {"Defaults": {}}
    print(f"{'node':<22} {'op_type':<12} {'MW':>5} {'MH':>4} {'SIMD':>5} {'PE':>4}")
    print("-" * 58)
    for i, (name, op, mw, mh) in enumerate(layers):
        f = by_dim.get((mw, mh)) or (FOLD[i] if i < len(FOLD) else None)
        if f is None:
            continue
        cfg[name] = {"SIMD": f[0], "PE": f[1]}
        print(f"{name:<22} {op:<12} {str(mw):>5} {str(mh):>4} {f[0]:5d} {f[1]:4d}")
    missing = [n for n, _, _, _ in layers if n not in cfg]
    if missing:
        raise RuntimeError(f"no folding for {missing} -- would run at SIMD=1 "
                           "and fail HLS codegen")
    json.dump(cfg, open(path, "w"), indent=2)
    print(f"\nwrote {os.path.basename(path)} ({len(cfg)-1} layers)")
    return cfg


print("defined discover_layers and write_folding")

defined discover_layers and write_folding


In [15]:


# ---------------------------------------------------------------- CELL E2 ---
# Export, estimate, folding. ~3 minutes.

tiny_e.cpu().eval()
export_qonnx(tiny_e, input_t=torch.zeros(1, 2, 32, 32), export_path=ONNX_E)
m = onnx.load(ONNX_E)
bip = {n.input[1] for n in m.graph.node if n.op_type == "BipolarQuant"}
for init in m.graph.initializer:
    if init.name in bip:
        init.raw_data = np.ones(1, dtype=np.float32).tobytes()
        init.float_data[:] = []
onnx.save(m, ONNX_E)
tiny_e.to(DEVICE)

from collections import Counter
ops = Counter(n.op_type for n in m.graph.node)
print(f"{os.path.basename(ONNX_E)}  nodes: {dict(ops)}")
print("Sigmoid in graph:", ops.get("Sigmoid", 0), "(want 0)\n")

vecs = torch.cat([encoder_input(p.obstacle_map,
                                index_to_one_hot(p.start, p.height, p.width),
                                index_to_one_hot(p.goal, p.height, p.width))
                  for p in list(eval_set)[:64]], 0)
np.savez(f"tiny_test_vectors{TAG_E}.npz",
         x=vecs.numpy(),
         y=torch.sigmoid(tiny_e(vecs.to(DEVICE))).detach().cpu().numpy())
print(f"saved tiny_test_vectors{TAG_E}.npz")

fresh(EST_E, EST_E + "_tmp")
build.build_dataflow_cfg(ONNX_E, make_cfg(EST_E, EST_E + "_tmp"))
FOLDING_E = os.path.join(WORKDIR, f"folding{TAG_E}.json")
write_folding(EST_E, FOLDING_E)

fresh(EST2_E, EST2_E + "_tmp")
build.build_dataflow_cfg(ONNX_E, make_cfg(EST2_E, EST2_E + "_tmp", folding=FOLDING_E))
cyc = load_report(EST2_E, "estimate_layer_cycles.json")
if cyc:
    worst = max(cyc.values())
    print(f"\nthroughput   : {worst*CLK_NS/1e6:.2f} ms")
    print(f"single-frame : {worst*(557868/147456)*CLK_NS/1e6:.2f} ms")
    print("Should match build D exactly -- same architecture, same folding.")



tiny_astare_w4a8.onnx  nodes: {'Quant': 9, 'Conv': 4, 'BatchNormalization': 4, 'Relu': 3}
Sigmoid in graph: 0 (want 0)

saved tiny_test_vectorse.npz
Building dataflow accelerator from $HOME/project/tiny_astare_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/tiny_este_tmp
Final outputs will be generated in $HOME/project/tiny_este
Build log is at $HOME/project/tiny_este/build_dataflow.log
Running step: step_qonnx_to_finn [1/11]
Running step: step_tidy_up [2/11]
Running step: step_streamline [3/11]
Running step: step_convert_to_hw [4/11]
Running step: step_create_dataflow_partition [5/11]
Running step: step_specialize_layers [6/11]
Running step: step_name_nodes [7/11]
Running step: step_target_fps_parallelization [8/11]
Running step: step_apply_folding_config [9/11]
Running step: step_minimize_bit_width [10/11]
Running step: step_generate_estimate_reports [11/11]
Completed successfully
layers from step_generate_estimate_reports.onnx

node                   op_type       

In [16]:

# ---------------------------------------------------------------- CELL E3 ---
# Full build. ~60 min. Run only if E1 showed p_opt recovering.

fresh(BUILD_E, BUILD_E + "_tmp")
print(f"full build started {time.strftime('%H:%M:%S')}\n")
t0 = time.time()
build.build_dataflow_cfg(ONNX_E, make_cfg(BUILD_E, BUILD_E + "_tmp",
                                          folding=FOLDING_E, bitfile=True))
print(f"\nBUILD COMPLETE in {(time.time()-t0)/60:.0f} min")

top = (load_report(BUILD_E, "post_synth_resources.json") or {}).get("(top)", {})
if top:
    print(f"\n{'Resource':<12} {'Used':>9} {'Available':>11} {'Util':>8}")
    print("-" * 44)
    for key, avail in PYNQ_Z2.items():
        used = top.get(key, top.get(key.replace("_36K", ""), 0))
        print(f"{key:<12} {used:9.0f} {avail:11d} {100*used/avail:7.1f}%")
    print("\nbuild D measured 35,266 LUT (66.3%) -- E should be near identical,")
    print("since only the trained weight values differ.")

drv = glob.glob(os.path.join(BUILD_E, "**", "driver.py"), recursive=True)
if drv:
    for line in open(drv[0]):
        if line.strip().startswith(('"idt"', '"odt"')):
            print(" ", line.strip())
    print("want odt INT8, as in build D")

json.dump({"tag": TAG_E, "channels": list(CHANNELS), "budget": MAC_BUDGET,
           "params": sum(p.numel() for p in tiny_e.parameters()),
           "no_sigmoid": True, "clamp": CLAMP, "final_l1": final_l1,
           "p_opt": re_.p_opt, "p_exp": re_.p_exp, "p_gen": re_.p_gen,
           "mac_cycle": mac_cycle, "cycles": bottleneck,
           "throughput_ms": bottleneck*CLK_NS/1e6,
           "latency_ms": bottleneck*(557868/147456)*CLK_NS/1e6,
           "est_lut": est_lut, "measured_lut": top.get("LUT", 0),
           "measured_lut_pct": 100*top.get("LUT", 0)/PYNQ_Z2["LUT"]},
          open(f"run{TAG_E}_summary.json", "w"), indent=1)

deploy = os.path.join(BUILD_E, "deploy")
if os.path.isdir(deploy):
    print(f"\nDEPLOYMENT PACKAGE: {deploy}")
    print("\nOn the board, expect build D's latency with build B's optimality.")
    print(f"  scp -r tiny_output{TAG_E}/deploy xilinx@<board-ip>:~/astar{TAG_E}")
    print(f"  scp tiny_test_vectors{TAG_E}.npz xilinx@<board-ip>:~/astar{TAG_E}/")
    print(f"  ssh xilinx@<board-ip> 'cd ~/astar{TAG_E} && "
          f"ln -s ../astard/neural_astar_core . && "
          f"ln -s ../astard/mazes_032_moore_c8.npz . && "
          f"cp ../astard/05_board.ipynb .'")


full build started 16:06:45

Building dataflow accelerator from $HOME/project/tiny_astare_w4a8.onnx
Intermediate outputs will be generated in $HOME/project/tiny_outpute_tmp
Final outputs will be generated in $HOME/project/tiny_outpute
Build log is at $HOME/project/tiny_outpute/build_dataflow.log
Running step: step_qonnx_to_finn [1/20]
Running step: step_tidy_up [2/20]
Running step: step_streamline [3/20]
Running step: step_convert_to_hw [4/20]
Running step: step_create_dataflow_partition [5/20]
Running step: step_specialize_layers [6/20]
Running step: step_name_nodes [7/20]
Running step: step_target_fps_parallelization [8/20]
Running step: step_apply_folding_config [9/20]
Running step: step_minimize_bit_width [10/20]
Running step: step_generate_estimate_reports [11/20]
Running step: step_hw_codegen [12/20]
Running step: step_hw_ipgen [13/20]
Running step: step_set_fifo_depths [14/20]
Running step: step_create_stitched_ip [15/20]
Running step: step_measure_rtlsim_performance [16/20]
Run